# 03 · Validating the mask→severity pipeline (coffee sets)
### *From Diagnosis to Decision* — ICA 2026

Notebook `02` *derives* severity from masks. Before that derivation can be used
as ground truth, it must be **validated against human severity labels**. Two
small coffee datasets make this possible — both < 2 GB, no authentication:

- **RoCoLe** — 1,560 robusta leaves, healthy/unhealthy + rust severity (L1–L4 by
  % affected area) **and leaf segmentation masks**. Because it has *both*
  severity labels and masks, it validates the mask→severity pipeline end to end.
  Mendeley DOI `10.17632/c5yvn32dzg.2`.
- **BRACOL** — 1,747 whole arabica leaves with 5 severity levels (by affected
  area) across miner, rust, brown-leaf-spot (Phoma) and cercospora. Mendeley
  DOI `10.17632/yy2k5y8mxg.1`. (Distinct from BRACOL's separate symptom-crop
  subset — use the whole-leaf set here.)

**Goal:** compute severity from RoCoLe's masks with the *same* function used on
PlantSeg, then measure how well it reproduces the human severity ordinal
(Spearman ρ, ordinal accuracy). A strong correlation licenses using PlantSeg-
derived severity as ground truth in the main experiments.

## 1 · Setup

In [ ]:
# --- Environment config: works on Google Colab AND locally --------------------
import os, sys, pathlib

def in_colab():
    return "google.colab" in sys.modules or os.path.exists("/content")

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/diagnosis-to-decision")
else:
    # local fallback: repo root (edit if you cloned elsewhere)
    PROJECT_ROOT = pathlib.Path(
        os.environ.get("ICA_PROJECT_ROOT", pathlib.Path.cwd().parents[0])
    )

DATA_RAW     = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_MAPPING = PROJECT_ROOT / "data" / "mapping"
FIGDIR       = PROJECT_ROOT / "reports" / "figures"
for p in (DATA_RAW, DATA_INTERIM, DATA_MAPPING, FIGDIR):
    p.mkdir(parents=True, exist_ok=True)

print("Colab:", in_colab())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_RAW exists:", DATA_RAW.exists())

In [ ]:
import numpy as np, pandas as pd, pathlib, json
import matplotlib.pyplot as plt
from PIL import Image
from skimage.filters import threshold_otsu
from skimage.color import rgb2hsv
from scipy.stats import spearmanr

RC = DATA_RAW / "rocole"
BR = DATA_RAW / "bracol"
print("RoCoLe present:", RC.exists() and any(RC.rglob("*")))
print("BRACOL present:", BR.exists() and any(BR.rglob("*")))

## 2 · Load RoCoLe labels + masks

RoCoLe ships annotations as JSON (polygon masks) plus a class/severity label per
leaf. Layout can vary by download; the loader below is defensive — adjust the
column/key names to match your extracted files, then the rest runs unchanged.

In [ ]:
def load_rocole():
    """Return DataFrame: image_path, human_severity (ordinal 0..4), mask (optional)."""
    # RoCoLe provides an annotations file (CSV or JSON). Try common names.
    csvs = list(RC.rglob("*.csv"))
    jsons = [p for p in RC.rglob("*.json") if "annotation" in p.name.lower() or "rocole" in p.name.lower()]
    df = None
    if csvs:
        df = pd.read_csv(csvs[0])
        print("loaded labels from", csvs[0].name, "| columns:", list(df.columns))
    elif jsons:
        data = json.loads(pathlib.Path(jsons[0]).read_text())
        print("loaded JSON annotations:", jsons[0].name, "- adapt parser to its schema")
    else:
        print("No RoCoLe annotation file found — download nb 00 / from Mendeley.")
    return df

rc = load_rocole()
if rc is not None:
    # Map RoCoLe rust levels to an ordinal 0..4. EDIT to match actual column names:
    #   e.g. classes: healthy, rust_1..rust_4, red_spider_mite
    display(rc.head())

## 3 · Reproduce mask→severity and correlate with human labels

Reuse the exact severity function from notebook `02` so the validation is
faithful. For each leaf: severity_hat = lesion_px / leaf_px from RoCoLe's mask;
compare to the human ordinal.

In [ ]:
def leaf_mask_otsu(img):
    a = np.asarray(img.convert("RGB"), dtype=np.float32)/255.0
    hsv = rgb2hsv(a)
    green = a[...,1] - 0.5*(a[...,0]+a[...,2])
    score = 0.5*(green-green.min())/(np.ptp(green)+1e-6) + 0.5*hsv[...,1]
    try: thr = threshold_otsu(score)
    except Exception: thr = score.mean()
    return score > thr

def severity_from_masks(img_path, lesion_mask):
    img = Image.open(img_path).convert("RGB")
    leaf = leaf_mask_otsu(img) | lesion_mask
    lp = int(leaf.sum())
    return np.nan if lp==0 else float(lesion_mask.sum())/lp

# --- Placeholder wiring: fill once RoCoLe paths/labels are resolved above ------
def run_validation(rc):
    """Expects rc with columns: image_path, lesion_mask_path, human_ordinal."""
    need = {"image_path","lesion_mask_path","human_ordinal"}
    if rc is None or not need.issubset(rc.columns):
        print("Wire up RoCoLe columns (image_path, lesion_mask_path, human_ordinal), then re-run.")
        return None
    rows=[]
    for _,r in rc.iterrows():
        m = np.asarray(Image.open(r["lesion_mask_path"]))
        m = (m[...,0] if m.ndim==3 else m) > 0
        s = severity_from_masks(r["image_path"], m)
        rows.append((r["human_ordinal"], s))
    v = pd.DataFrame(rows, columns=["human_ordinal","severity_hat"]).dropna()
    rho,p = spearmanr(v["human_ordinal"], v["severity_hat"])
    print(f"Spearman rho = {rho:.3f} (p={p:.1e}) over {len(v)} leaves")
    # ordinal accuracy after binning severity_hat into the same #levels
    q = pd.qcut(v["severity_hat"], q=v["human_ordinal"].nunique(), labels=False, duplicates="drop")
    acc = (q == v["human_ordinal"].rank(method="dense").astype(int)-1).mean()
    print(f"binned ordinal agreement ~ {acc:.2f}")
    v.to_csv(DATA_INTERIM/"rocole_severity_validation.csv", index=False)
    return v, rho

res = run_validation(rc)

In [ ]:
# Scatter: human ordinal vs derived severity (fills in once wired)
if isinstance(res, tuple):
    v,rho = res
    fig,ax=plt.subplots(figsize=(6,4))
    ax.scatter(v["human_ordinal"]+np.random.default_rng(0).normal(0,0.05,len(v)),
               v["severity_hat"], s=10, alpha=0.4, color="#B279A2")
    ax.set_xlabel("human severity ordinal"); ax.set_ylabel("derived severity (lesion/leaf)")
    ax.set_title(f"RoCoLe: mask-derived vs human severity (Spearman {rho:.2f})")
    plt.tight_layout(); fig.savefig(FIGDIR/"rocole_validation.png"); plt.show()

## 4 · BRACOL cross-check (severity labels, no masks)

BRACOL has severity labels but not per-lesion masks, so it can't validate the
*mask* step — but it can check whether a **learned** severity regressor trained
on PlantSeg-derived severity ranks BRACOL leaves correctly (transfer sanity
check). Left as a scoped extension.

In [ ]:
print("BRACOL: use as an out-of-distribution ranking check for a severity model "
      "trained on PlantSeg-derived targets. Load whole-leaf images + severity CSV, "
      "predict, and report Spearman rho vs BRACOL's human severity.")

---
### For the paper
- Report **Spearman ρ** (and ordinal agreement) between mask-derived and human
  severity on RoCoLe. This is the evidence that PlantSeg-derived severity is a
  usable proxy — without it, the severity→action mapping rests on an unvalidated
  assumption a reviewer will challenge.
- If ρ is weak, that is still a finding: it says the zero-cost Otsu leaf estimate
  is the bottleneck and motivates a learned leaf segmenter.